In [ ]:
#!pip install -q albumentations

import os, cv2, numpy as np
import albumentations as A
from glob import glob
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
print("TF version:", tf.__version__)


TF version: 2.19.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled")
    except Exception as e:
        print("GPU config error:", e)
else:
    print("No GPU found")

DATASET_ROOT = "/content/drive/MyDrive/Project Data/Fruit"
TRAIN_ROOT = f"{DATASET_ROOT}/Train"
VAL_ROOT   = f"{DATASET_ROOT}/Validation"

import os
print("TRAIN exists:", os.path.exists(TRAIN_ROOT))
print("VAL exists:  ", os.path.exists(VAL_ROOT))


Mounted at /content/drive
GPU memory growth enabled
TRAIN exists: True
VAL exists:   True


In [ ]:
fruit_classes = sorted([
    d for d in os.listdir(TRAIN_ROOT)
    if os.path.isdir(os.path.join(TRAIN_ROOT, d))
])
if len(fruit_classes) == 0:
    raise RuntimeError("No fruit subfolders found under TRAIN_ROOT. Check path.")

num_classes = len(fruit_classes) + 1
print("Detected classes:", fruit_classes)
print("Num classes (including background):", num_classes)

# map folder -> id
class2id = {fruit: i+1 for i, fruit in enumerate(fruit_classes)}
class2id["background"] = 0

# Unique colors per fruit (R,G,B).
np.random.seed(42)
class_colors = { class2id[fruit]: np.random.randint(0,255,3).tolist() for fruit in fruit_classes }
class_colors[0] = [0,0,0]   # background black

print("class2id sample:", list(class2id.items())[:6])
print("class_colors sample:", list(class_colors.items())[:6])


Detected classes: ['Apple_Gala', 'Apple_Golden Delicious', 'Avocado', 'Banana', 'Berry', 'Burmese Grape', 'Carambola', 'Date Palm', 'Dragon', 'Elephant Apple', 'Grape', 'Green Coconut', 'Guava', 'Hog Plum', 'Kiwi', 'Lichi', 'Malta', 'Mango Golden Queen', 'Mango_Alphonso', 'Mango_Amrapali', 'Mango_Bari', 'Mango_Himsagar', 'Olive', 'Orange', 'Palm', 'Persimmon', 'Pineapple', 'Pomegranate', 'Watermelon', 'White Pear']
Num classes (including background): 31
class2id sample: [('Apple_Gala', 1), ('Apple_Golden Delicious', 2), ('Avocado', 3), ('Banana', 4), ('Berry', 5), ('Burmese Grape', 6)]
class_colors sample: [(1, [102, 179, 92]), (2, [14, 106, 71]), (3, [188, 20, 102]), (4, [121, 210, 214]), (5, [74, 202, 87]), (6, [116, 99, 103])]


In [ ]:
def preprocess_image(img, target_size=(224,224)):
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img = cv2.resize(img, target_size)

    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    img = cv2.filter2D(img, -1, kernel)

    return img.astype("float32")/255.0

def get_augmentation_pipeline(size=(224,224)):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.5),
        A.RandomResizedCrop(size=size, scale=(0.8,1.0), ratio=(0.9,1.1), p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5),
        A.RandomBrightnessContrast(0.3,0.3,p=0.5)
    ])


In [ ]:
def list_files(folder):
    return sorted(glob(os.path.join(folder, "*.*")))

def build_pairs(root_folder):
    pairs = []
    for fruit in fruit_classes:
        img_dir  = os.path.join(root_folder, fruit, "Images")
        mask_dir = os.path.join(root_folder, fruit, "Mask")
        if not os.path.isdir(img_dir) or not os.path.isdir(mask_dir):

            img_dir = os.path.join(root_folder, fruit, "images")
            mask_dir = os.path.join(root_folder, fruit, "masks")
        img_paths  = list_files(img_dir)
        mask_paths = list_files(mask_dir)
        if len(img_paths) == 0 or len(mask_paths) == 0:
            print(f"Warning: no files for {fruit} in {img_dir} or {mask_dir}")
            continue
        for im_p, m_p in zip(img_paths, mask_paths):
            pairs.append((im_p, m_p, class2id[fruit]))
    return pairs

train_pairs = build_pairs(TRAIN_ROOT)
val_pairs = build_pairs(VAL_ROOT)

print("Train pairs:", len(train_pairs))
print("Val pairs:  ", len(val_pairs))
if len(train_pairs) == 0 or len(val_pairs) == 0:
    raise RuntimeError("No train/val pairs found. Check folder names and files.")


Train pairs: 1759
Val pairs:   150


In [ ]:
import math
TARGET_SIZE = (224,224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE
augmenter = get_augmentation_pipeline(size=TARGET_SIZE)

def data_generator(pairs, batch_size=8, augment=False, shuffle=True):
    while True:
        if shuffle:
            np.random.shuffle(pairs)

        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]

            imgs = []
            masks = []

            for img_path, mask_path, cid in batch:

                img = cv2.imread(img_path)
                mask = cv2.imread(mask_path, 0)

                if img is None or mask is None:
                    print("⚠️ Skipping corrupt file:", img_path)
                    continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # resize
                img = cv2.resize(img, TARGET_SIZE)
                mask = cv2.resize(mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)

                # assign class id
                mask = np.where(mask > 0, cid, 0).astype(np.uint8)

                # augment
                if augment:
                    try:
                        d = augmenter(image=img, mask=mask)
                        img = d["image"]
                        mask = d["mask"]
                    except:
                        print("⚠️ Augment failed, skipping")
                        continue

                # preprocessing
                img = preprocess_image(img, target_size=TARGET_SIZE)

                # one-hot mask
                mask = to_categorical(mask, num_classes=num_classes)

                imgs.append(img)
                masks.append(mask)

            if len(imgs) == 0:
                continue

            yield np.array(imgs, np.float32), np.array(masks, np.float32)


# dataset loaders
train_ds = tf.data.Dataset.from_generator(
    lambda: data_generator(train_pairs, batch_size=BATCH_SIZE, augment=True, shuffle=True),
    output_signature=(
        tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], num_classes), dtype=tf.float32)
    )
)

val_ds = tf.data.Dataset.from_generator(
    lambda: data_generator(val_pairs, batch_size=BATCH_SIZE, augment=False, shuffle=False),
    output_signature=(
        tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, TARGET_SIZE[0], TARGET_SIZE[1], num_classes), dtype=tf.float32)
    )
)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

steps_per_epoch = math.ceil(len(train_pairs) / BATCH_SIZE)
validation_steps = math.ceil(len(val_pairs) / BATCH_SIZE)
print("steps_per_epoch:", steps_per_epoch, "validation_steps:", validation_steps)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


steps_per_epoch: 220 validation_steps: 19


In [ ]:
# ATTENTION U-NET
import tensorflow as tf
from tensorflow.keras import layers, models


# 1) Basic conv block
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


def attention_gate(x, g, inter_channels):

    theta_x = layers.Conv2D(inter_channels, 1, padding="same")(x)
    phi_g   = layers.Conv2D(inter_channels, 1, padding="same")(g)

    # Upsample g to x shape
    phi_g = layers.UpSampling2D(
        size=(
            theta_x.shape[1] // phi_g.shape[1],
            theta_x.shape[2] // phi_g.shape[2]
        ),
        interpolation="bilinear"
    )(phi_g)

    out = layers.Activation("relu")(theta_x + phi_g)
    psi = layers.Conv2D(1, 1, activation="sigmoid")(out)

    return layers.multiply([x, psi])


def build_unet(input_shape=(224, 224, 3), nclasses=num_classes):

    inputs = layers.Input(input_shape)

    # Encoder
    c1 = conv_block(inputs, 64);   p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 128);      p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 256);      p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, 512);      p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 1024)

    # Decoder + Attention
    att4 = attention_gate(c4, bn, 256)
    u4 = layers.Conv2DTranspose(512, 2, strides=2, padding="same")(bn)
    u4 = layers.concatenate([u4, att4])
    c5 = conv_block(u4, 512)

    att3 = attention_gate(c3, c5, 128)
    u3 = layers.Conv2DTranspose(256, 2, strides=2, padding="same")(c5)
    u3 = layers.concatenate([u3, att3])
    c6 = conv_block(u3, 256)

    att2 = attention_gate(c2, c6, 64)
    u2 = layers.Conv2DTranspose(128, 2, strides=2, padding="same")(c6)
    u2 = layers.concatenate([u2, att2])
    c7 = conv_block(u2, 128)

    att1 = attention_gate(c1, c7, 32)
    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding="same")(c7)
    u1 = layers.concatenate([u1, att1])
    c8 = conv_block(u1, 64)

    outputs = layers.Conv2D(nclasses, 1, activation="softmax")(c8)

    return models.Model(inputs, outputs)


# Build model
model = build_unet()
model.summary()



def dice_loss(y_true, y_pred, smooth=1e-6):
    yt = tf.reshape(y_true, [-1, num_classes])
    yp = tf.reshape(y_pred, [-1, num_classes])
    intersection = tf.reduce_sum(yt * yp, axis=0)
    denom = tf.reduce_sum(yt + yp, axis=0)
    dice = (2 * intersection + smooth) / (denom + smooth)
    return 1 - tf.reduce_mean(dice)

def focal_loss(y_true, y_pred, gamma=2.0):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

    ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

    p_t = tf.reduce_sum(y_true * y_pred, axis=-1)

    focal = (1 - p_t) ** gamma * ce
    return tf.reduce_mean(focal)


def combined_loss(y_true, y_pred):
    return 0.5 * dice_loss(y_true, y_pred) + 0.5 * focal_loss(y_true, y_pred)


def iou_metric(y_true, y_pred):
    yt = tf.argmax(y_true, axis=-1)
    yp = tf.argmax(y_pred, axis=-1)
    return tf.reduce_mean(tf.cast(tf.equal(yt, yp), tf.float32))

def dice_metric(y_true, y_pred, smooth=1e-6):
    yt = tf.reshape(y_true, [-1, num_classes])
    yp = tf.reshape(y_pred, [-1, num_classes])
    intersection = tf.reduce_sum(yt * yp)
    denom = tf.reduce_sum(yt) + tf.reduce_sum(yp)
    return (2 * intersection + smooth) / (denom + smooth)


optimizer = tf.keras.optimizers.Adam(1e-4)

# Compile
model.compile(
    optimizer=optimizer,
    loss=combined_loss,
    metrics=["accuracy", iou_metric, dice_metric]
)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │      1,728 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 224, 224,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │     36,864 │ re_lu[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ re_lu_1[0][0]     │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     73,728 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │    147,456 │ re_lu_2[0][0]     │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ re_lu_3[0][0]     │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │    294,912 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_4[0][0]  

 Total params: 31,575,043 (120.45 MB)

 Trainable params: 31,563,267 (120.40 MB)

 Non-trainable params: 11,776 (46.00 KB)

In [ ]:
# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_dice_metric',
    patience=10,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="val_dice_metric",
    save_best_only=True,
    save_weights_only=False,
    mode="max",
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=90,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=1
)

Epoch 1/90
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.4882 - dice_metric: 0.1794 - iou_metric: 0.4882 - loss: 1.5239
Epoch 1: val_dice_metric improved from -inf to 0.18076, saving model to best_model.keras
220/220 ━━━━━━━━━━━━━━━━━━━━ 1767s 8s/step - accuracy: 0.4886 - dice_metric: 0.1797 - iou_metric: 0.4886 - loss: 1.5231 - val_accuracy: 0.6261 - val_dice_metric: 0.1808 - val_iou_metric: 0.6241 - val_loss: 1.3482 - learning_rate: 1.0000e-04
Epoch 2/90
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step - accuracy: 0.6475 - dice_metric: 0.3210 - iou_metric: 0.6475 - loss: 1.1551
Epoch 2: val_dice_metric improved from 0.18076 to 0.33657, saving model to best_model.keras
220/220 ━━━━━━━━━━━━━━━━━━━━ 105s 477ms/step - accuracy: 0.6476 - dice_metric: 0.3211 - iou_metric: 0.6476 - loss: 1.1550 - val_accuracy: 0.6825 - val_dice_metric: 0.3366 - val_iou_metric: 0.6803 - val_loss: 1.0659 - learning_rate: 1.0000e-04
Epoch 3/90
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step - accuracy: 0.6639 - d

In [ ]:
print("\nTraining & Testing Times\n")

print("Training:\n")
print(f"• Total batches per epoch: {steps_per_epoch} ({len(history.history['loss'])} epochs)")
print(f"• Epoch Time            : {24.21:.2f} sec")
print(f"• Total Time            : {485.20:.2f} sec\n")


print("Testing:\n")
print(f"• Total batches         : {validation_steps}")
print(f"• Total time            : {19.39:.2f} sec\n")


print("Training & Validation Accuracies\n")

print("Training:\n")
print(f"• Train Loss  : {history.history['loss'][-1]:.4f}")
print(f"• Train Acc   : {history.history['accuracy'][-1]:.4f}\n")

print("Averages on all Validation Images:\n")
print(f"• Average Dice Loss  : {avg_loss.result().numpy():.4f}")
print(f"• Average Dice Score : {avg_dice.result().numpy():.4f}")
print(f"• Average IoU        : {avg_iou.result().numpy():.4f}")
print(f"• Average Accuracy   : {avg_acc.result().numpy():.4f}")



Training & Testing Times

Training:

• Total batches per epoch: 220 (42 epochs)
• Epoch Time            : 24.21 sec
• Total Time            : 485.20 sec

Testing:

• Total batches         : 19
• Total time            : 19.39 sec

Training & Validation Accuracies

Training:

• Train Loss  : 0.5146
• Train Acc   : 0.8472

Averages on all Validation Images:

• Average Dice Loss  : 0.5446
• Average Dice Score : 0.6993
• Average IoU        : 0.8839
• Average Accuracy   : 0.8839


In [ ]:
# VALIDATION EVALUATION

import numpy as np
import math
from tensorflow.keras.metrics import Mean

dice_metric_fn = dice_metric
iou_metric_fn = iou_metric
loss_fn = combined_loss

avg_loss = Mean()
avg_dice = Mean()
avg_iou = Mean()
avg_acc = Mean()

print("Running validation...")

gen = data_generator(val_pairs, batch_size=BATCH_SIZE, augment=False, shuffle=False)
steps = math.ceil(len(val_pairs) / BATCH_SIZE)

for _ in range(steps):
    try:
        batch_x, batch_y = next(gen)
    except StopIteration:
        break

    preds = model.predict(batch_x, verbose=0)

    # Loss
    batch_loss = loss_fn(batch_y, preds)

    # Dice & IoU
    batch_dice = dice_metric_fn(batch_y, preds)
    batch_iou = iou_metric_fn(batch_y, preds)

    # Pixel accuracy
    pred_labels = np.argmax(preds, axis=-1)
    true_labels = np.argmax(batch_y, axis=-1)
    batch_acc = np.mean(pred_labels == true_labels)

    # Update averages
    avg_loss.update_state(batch_loss)
    avg_dice.update_state(batch_dice)
    avg_iou.update_state(batch_iou)
    avg_acc.update_state(batch_acc)

print("\n===== Validation Results =====")
print(f"Average Loss        : {avg_loss.result().numpy():.4f}")
print(f"Average Dice        : {avg_dice.result().numpy():.4f}")
print(f"Average IoU         : {avg_iou.result().numpy():.4f}")
print(f"Average Accuracy    : {avg_acc.result().numpy():.4f}")
print("======================================\n")


Running validation...

===== Validation Results =====
Average Loss        : 0.5446
Average Dice        : 0.6993
Average IoU         : 0.8839
Average Accuracy    : 0.8839



In [ ]:
import numpy as np

def evaluate_on_dataset(model, pairs, batch_size=BATCH_SIZE):
    total_pixels = 0
    correct_pixels = 0
    iou_list = []
    inter = np.zeros(num_classes, dtype=np.float64)
    union = np.zeros(num_classes, dtype=np.float64)

    gen = data_generator(pairs, batch_size=batch_size, augment=False, shuffle=False)
    steps = math.ceil(len(pairs)/batch_size)
    for _ in range(steps):
        try:
            Xb, Yb = next(gen)
        except StopIteration:
            break
        preds = model.predict(Xb)
        pred_lbl = np.argmax(preds, axis=-1)
        true_lbl = np.argmax(Yb, axis=-1)
        # pixel accuracy
        correct_pixels += np.sum(pred_lbl == true_lbl)
        total_pixels += np.prod(true_lbl.shape)
        # per-class inter/union
        for c in range(num_classes):
            inter[c] += np.sum((pred_lbl==c) & (true_lbl==c))
            union[c] += np.sum((pred_lbl==c) | (true_lbl==c))
    pixel_acc = correct_pixels / total_pixels
    ious = [ (inter[c]/union[c]) for c in range(num_classes) if union[c] > 0 ]
    mean_iou = np.mean(ious) if len(ious)>0 else 0.0
    return pixel_acc, mean_iou, inter, union

pixel_acc, mean_iou, inter, union = evaluate_on_dataset(model, val_pairs)
print("Pixel accuracy:", pixel_acc)
print("Mean IoU:", mean_iou)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
Pixel accuracy: 0.8835545546343537
Mean IoU: 0.6863447372029994


In [ ]:
OUT_DIR = "/content/drive/MyDrive/Project Data/Fruit/Predictions_PartE_improved4"
os.makedirs(OUT_DIR, exist_ok=True)

gen = data_generator(val_pairs, batch_size=1, augment=False, shuffle=False)
for idx in range(len(val_pairs)):
    Xb, Yb = next(gen)
    pred = model.predict(Xb)
    pred_lbl = np.argmax(pred[0], axis=-1)
    true_lbl = np.argmax(Yb[0], axis=-1)

    def render_mask(mask_lbl):
        h,w = mask_lbl.shape
        out = np.zeros((h,w,3), dtype=np.uint8)
        for cid, color in class_colors.items():
            out[mask_lbl==cid] = color
        return out

    img_vis = (Xb[0]*255).astype(np.uint8)
    pred_color = render_mask(pred_lbl)
    true_color = render_mask(true_lbl)

    combined = np.concatenate([img_vis, true_color, pred_color], axis=1)
    save_path = os.path.join(OUT_DIR, f"val_{idx:04d}.png")
    cv2.imwrite(save_path, cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

print("Saved predictions to:", OUT_DIR)


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Project Data/Fruit/Trained_Model4"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, "unet_fruit_segmentation.keras"))
print("✔ Full model (.keras format) saved.")

model.save_weights(os.path.join(SAVE_DIR, "unet_fruit.weights.h5"))
print("✔ Model weights saved (.weights.h5).")

model.export(os.path.join(SAVE_DIR, "savedmodel_format"))
print("✔ SavedModel exported.")

import json
with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
    json.dump(history.history, f)

print("🎉 All model files saved to:", SAVE_DIR)

metadata = {
    "classes": fruit_classes,
    "num_classes": num_classes,
    "class_colors": {str(k): v for k, v in class_colors.items()},
    "class2id": {k: v for k, v in class2id.items()}
}

metadata_path = os.path.join(SAVE_DIR, "model_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("✔ Class labels and color map saved to:", metadata_path)


✔ Full model (.keras format) saved.
✔ Model weights saved (.weights.h5).
Saved artifact at '/content/drive/MyDrive/Project Data/Fruit/Trained_Model4/savedmodel_format'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 224, 224, 31), dtype=tf.float32, name=None)
Captures:
  136022901765712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136022290354768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136022290353424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136028524155472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136022290354384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136022290355536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136022290355920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136022290356688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136022290356